In [ ]:
library("tidyverse")


In [ ]:
fileDir <- "./Data/"
fileName <- "pro_data_common_val_Assay.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_data <- temp


fileDir <- "./Data/"
fileName <- "pro_sample_val.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path,row.names = 1)
head(temp)
pro_sample <- temp

fileDir <- "./Data/"
fileName <- "map_OID_Uni_Sym.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
pro_analyte <- temp

fileDir <- "./Data/"
fileName <- "DEP_list.csv"

Path <- str_c(fileDir, fileName)
temp <- read.csv(Path, row.names = 1)
head(temp)
DEPlist <- temp

In [ ]:
y <- log10(pro_sample[["CA19_9"]])
head(y)

In [ ]:
vars <- c(
  "progress",
  "sex",
  "fvage",
  "CaCo_Status",
  "smoker",
  "py",
  "drinker",
  "quant_drink_gr",
  "bmi",
  "hx_dm"
)


results <- data.frame()

for (var in vars) {
  
  x <- pro_sample[[var]]
  
  dat <- data.frame(
    logvalue = y,
    x = x
  )
  
  dat <- dat[complete.cases(dat), ]
  
  # 数値変数
  if (is.numeric(x)) {
    
    test <- cor.test(
      dat$logvalue,
      dat$x,
      method = "spearman"
    )
    
    results <- rbind(
      results,
      data.frame(
        Variable = var,
        Method = "Spearman",
        Statistic = unname(test$estimate),
        P_value = test$p.value,
        N = nrow(dat)
      )
    )
    
  } else {
    
    dat$x <- as.factor(dat$x)
    
    # 2群
    if (nlevels(dat$x) == 2) {
      
      test <- wilcox.test(
        logvalue ~ x,
        data = dat
      )
      
      results <- rbind(
        results,
        data.frame(
          Variable = var,
          Method = "Wilcoxon",
          Statistic = NA,
          P_value = test$p.value,
          N = nrow(dat)
        )
      )
      
    # 3群以上
    } else if (nlevels(dat$x) > 2) {
      
      test <- kruskal.test(
        logvalue ~ x,
        data = dat
      )
      
      results <- rbind(
        results,
        data.frame(
          Variable = var,
          Method = "Kruskal-Wallis",
          Statistic = NA,
          P_value = test$p.value,
          N = nrow(dat)
        )
      )
    }
  }
}

# 多重比較補正
results$FDR <- p.adjust(results$P_value, method = "BH")

# P値の小さい順
results <- results[order(results$P_value), ]

results

In [ ]:
library(ggplot2)

plot_data <- data.frame(
  CaCo_Status = factor(
    pro_sample$CaCo_Status,
    levels = c(0, 1),
    labels = c("Non-cancer", "Cancer")
  ),
  logvalue = log10(pro_sample[["CA19_9"]])
)

plot_data <- plot_data[complete.cases(plot_data), ]

ggplot(plot_data, aes(x = CaCo_Status, y = logvalue)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.15, alpha = 0.5) +
  labs(
    x = "Cancer status",
    y = "log10(CA19-9)"
  ) +
  theme_classic()